# Bottleneck splitting

Implementation of the bottleneck-detection and cutting steps (2-3) of:

> Wang, H., Zhang, H., & Ray, N. (2012). Clump splitting via bottleneck detection and shape classification. *Pattern Recognition*, 45(7), 2780-2787.

Both chi-shape and chi-outline return a single simple polygon per connected region of points -- but a "region" can itself be a clump of several touching/overlapping objects (e.g. touching fruit in a cluster) that merged into one polygon. `BottleneckSplit` takes such a polygon and recursively cuts it at its narrowest "neck" -- the pair of boundary points minimizing
```
Es(A, B) = dist(A, B) / min(arc(A, B), arc(B, A))
```
(Eq. 1-2 of the paper) -- for as long as a pluggable `SplitCriterion` says the result still looks like a clump. Step 3 of the paper finds the cut via a weighted shortest path over image-intensity edge strength; we only have polygon geometry here (no source image), which the paper itself says the weighted path degenerates to: a straight-line cut between the two points.

**Important, from empirical testing below**: the original paper's shape-classification step is an offline-trained, application-specific SVM -- we don't have that training data, so this package ships simple geometric substitutes instead (see `SplitCriterion` subclasses). Those substitutes only separate "genuinely touching objects" from "one noisy blob" reliably when the input polygon was built with a moderate-to-high chi/chi_factor (~0.5+); at low chi, chi-shape/chi-outline carve so many small non-convex notches into *any* point cloud that a single unmerged blob can look just as "non-convex" as a real clump. Every slider below includes this parameter so you can feel that tradeoff directly.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

from concavehull import (
    concavehull,
    ChiOutline,
    BottleneckSplit,
    BottleneckCostCriterion,
    ConvexDeficitCriterion,
    ConvexHullL2Criterion,
)


def plot_pieces(ax, pts, pieces, title=None, point_size=6):
    ax.scatter(pts[:, 0], pts[:, 1], s=point_size, alpha=0.3, color="gray")
    colors = plt.cm.tab10.colors
    for i, piece in enumerate(pieces):
        closed = np.vstack([piece, piece[:1]])
        ax.plot(closed[:, 0], closed[:, 1], color=colors[i % len(colors)], linewidth=2)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    if title:
        ax.set_title(title)

## Example 1: Splitting a merged clump

Two overlapping blobs, merged into one `concavehull()` polygon, split back apart by `BottleneckSplit` using the default criterion (`BottleneckCostCriterion`).

In [2]:
@interact(
    n_per_blob=IntSlider(value=200, min=50, max=600, step=25, description="n_per_blob",
                          continuous_update=False),
    separation=FloatSlider(value=4.2, min=2.0, max=8.0, step=0.2, description="separation",
                            continuous_update=False),
    chi_factor=FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description="chi_factor",
                            continuous_update=False),
    cost_threshold=FloatSlider(value=0.45, min=0.05, max=0.95, step=0.05, description="cost_threshold",
                                continuous_update=False),
    min_piece_area=FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description="min_piece_area",
                                continuous_update=False),
)
def show_split(n_per_blob, separation, chi_factor, cost_threshold, min_piece_area):
    rng = np.random.default_rng(0)
    blob_a = rng.normal(loc=[0, 0], scale=1.0, size=(n_per_blob, 2))
    blob_b = rng.normal(loc=[separation, 0], scale=1.0, size=(n_per_blob, 2))
    pts = np.vstack([blob_a, blob_b])

    hull = concavehull(pts, chi_factor=chi_factor)
    bs = BottleneckSplit(BottleneckCostCriterion(cost_threshold), min_piece_area=min_piece_area)
    pieces = bs.split(hull)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    plot_pieces(axes[0], pts, [hull], title=f"concavehull() (chi_factor={chi_factor:.2f})")
    plot_pieces(axes[1], pts, pieces, title=f"BottleneckSplit -> {len(pieces)} piece(s)")
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=200, continuous_update=False, description='n_per_blob', max=600, min=50,…

## Example 2: Comparing criteria

`SplitCriterion` is a pluggable, swappable piece -- three concrete implementations ship with this package:

- `BottleneckCostCriterion` (default): judges the single best candidate neck's own cost. A local signal, less confused by boundary noise elsewhere.
- `ConvexDeficitCriterion`: judges the whole polygon's area lost to concavity, `1 - solidity`, naturally bounded to `[0, 1)`.
- `ConvexHullL2Criterion`: the chi-shape paper's own L2 error norm (Sec. 6.1), applied between the polygon and its own convex hull. **Not** bounded to `[0, 1]` -- it is `1/solidity - 1`, which grows without bound as the polygon thins relative to its hull, so its threshold lives on a different scale than the other two.

All three are judging the *same* polygon here, so you can directly compare how each threshold scale behaves.

In [3]:
@interact(
    n_per_blob=IntSlider(value=200, min=50, max=600, step=25, description="n_per_blob",
                          continuous_update=False),
    separation=FloatSlider(value=4.2, min=2.0, max=8.0, step=0.2, description="separation",
                            continuous_update=False),
    chi_factor=FloatSlider(value=0.6, min=0.1, max=1.0, step=0.05, description="chi_factor",
                            continuous_update=False),
    cost_threshold=FloatSlider(value=0.45, min=0.05, max=0.95, step=0.05, description="cost_thr",
                                continuous_update=False),
    deficit_threshold=FloatSlider(value=0.15, min=0.01, max=0.6, step=0.01, description="deficit_thr",
                                   continuous_update=False),
    l2_threshold=FloatSlider(value=0.18, min=0.02, max=1.5, step=0.02, description="l2_thr",
                              continuous_update=False),
    min_piece_area=FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description="min_piece_area",
                                continuous_update=False),
)
def compare_criteria(n_per_blob, separation, chi_factor, cost_threshold, deficit_threshold, l2_threshold,
                      min_piece_area):
    rng = np.random.default_rng(0)
    blob_a = rng.normal(loc=[0, 0], scale=1.0, size=(n_per_blob, 2))
    blob_b = rng.normal(loc=[separation, 0], scale=1.0, size=(n_per_blob, 2))
    pts = np.vstack([blob_a, blob_b])
    hull = concavehull(pts, chi_factor=chi_factor)

    criteria = [
        ("BottleneckCostCriterion", BottleneckCostCriterion(cost_threshold)),
        ("ConvexDeficitCriterion", ConvexDeficitCriterion(deficit_threshold)),
        ("ConvexHullL2Criterion", ConvexHullL2Criterion(l2_threshold)),
    ]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (name, crit) in zip(axes, criteria):
        pieces = BottleneckSplit(crit, min_piece_area=min_piece_area).split(hull)
        plot_pieces(ax, pts, pieces, title=f"{name}\n-> {len(pieces)} piece(s)")
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=200, continuous_update=False, description='n_per_blob', max=600, min=50,…

## Example 3: Full pipeline -- chi-outline then bottleneck splitting

The intended end-to-end use: run chi-outline on a scene with multiple (possibly clumped) objects plus outliers, then run `BottleneckSplit.split_all()` over every returned polygon to separate any remaining merged clumps. Each of the 3 base clusters below is itself a pair of overlapping sub-blobs, simulating two touching objects per cluster.

Note the default `chi=3.0` here is much higher than chi-outline's other examples use (~1.0-1.3) or than chi-shape's `chi_factor` scale ([0, 1]) would suggest -- `chi` is an unbounded multiplier on each component's edge-length standard deviation (lambda = mu + chi*sigma), not a normalized fraction, so it is not directly comparable to chi_factor. In testing, chi-outline's boundaries stayed much jaggier than chi-shape's at a nominally "comparable" chi, which was enough to make BottleneckSplit's default threshold over-split badly; both `chi` and `cost_threshold` needed retuning together to get clean splits here. This is the calibration-coupling the warning at the top of this notebook describes, made concrete.

In [4]:
@interact(
    n_per_subblob=IntSlider(value=150, min=40, max=400, step=10, description="n_per_subblob",
                             continuous_update=False),
    sub_separation=FloatSlider(value=3.8, min=2.0, max=6.0, step=0.2, description="sub_separation",
                                continuous_update=False),
    chi=FloatSlider(value=3.0, min=0.5, max=4.5, step=0.1, description="chi (outline)",
                     continuous_update=False),
    cost_threshold=FloatSlider(value=0.25, min=0.05, max=0.95, step=0.05, description="cost_thr (split)",
                                continuous_update=False),
    min_piece_area=FloatSlider(value=0.0, min=0.0, max=5.0, step=0.1, description="min_piece_area",
                                continuous_update=False),
)
def show_pipeline(n_per_subblob, sub_separation, chi, cost_threshold, min_piece_area):
    rng = np.random.default_rng(1)
    centers = [(0, 0), (12, 0), (6, 10)]
    clusters = []
    for cx, cy in centers:
        a = rng.normal(loc=[cx, cy], scale=1.0, size=(n_per_subblob, 2))
        b = rng.normal(loc=[cx + sub_separation, cy], scale=1.0, size=(n_per_subblob, 2))
        clusters.append(np.vstack([a, b]))
    outliers = rng.uniform(low=[-4, -4], high=[20, 14], size=(25, 2))
    pts = np.vstack(clusters + [outliers])

    co = ChiOutline(pts, chi=chi)
    regions = co.compute()

    bs = BottleneckSplit(BottleneckCostCriterion(cost_threshold), min_piece_area=min_piece_area)
    pieces = bs.split_all(regions)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    plot_pieces(axes[0], pts, regions, title=f"ChiOutline.compute() -> {len(regions)} region(s)")
    plot_pieces(axes[1], pts, pieces, title=f"+ BottleneckSplit.split_all() -> {len(pieces)} piece(s)")
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=150, continuous_update=False, description='n_per_subblob', max=400, min=…

## Notes

- `BottleneckSplit` and `SplitCriterion` operate purely on 2D polygon geometry. The original paper's method was developed and validated on smooth boundaries extracted from binarized microscopy/ore images -- considerably smoother than a sparse point cloud's chi-shape/chi-outline boundary. If your real pipeline produces image-contour polygons instead of point-cloud ones, expect these criteria to behave better than the empirical numbers quoted above, which were measured on point clouds.
- `find_bottleneck` is O(n^2) in the polygon's vertex count per split, and splitting recurses up to `max_depth` times -- fine for the chi-shape/chi-outline boundary sizes used here (tens to low hundreds of vertices), worth knowing for much larger polygons.